# Stage 6 — Sub-clustering (Cross-Machine Merge)

**Input**: `adata_pooled.h5ad` from Stage 5 for **all machines**  
**Output**: Final AnnData, final heatmap, per-ROI CSVs with full lineage  
**Scope**: All machines merged — subclusters each meta-cluster group independently

For each meta-cluster group (e.g., "T Cells"), the pipeline:
1. Extracts all cells across machines
2. Re-normalises and runs Leiden at multiple resolutions
3. Presents dotplot/heatmap candidates for visual comparison
4. Human selects the best resolution per group
5. Final labels are backtracked to individual ROIs

In [1]:
import numpy as np
import pandas as pd
import anndata as ad
from IPython.display import display

from pipeline.config import load_config
from pipeline.io import (
    load_adata,
    save_adata,
    save_roi_assignments,
    validate_stage_inputs,
)
from pipeline.clustering import subcluster_at_resolutions, prepare_cluster_means
from pipeline.visualization import plot_dotplot, plot_heatmap, plot_spatial
from pipeline.widgets import create_batch_resolution_picker

cfg = load_config("config.yaml")
np.random.seed(cfg.clustering.random_seed)

## Load & Merge All Machines

Concatenate pooled AnnData from Stage 5 across all machines.

In [2]:
# Load pooled AnnData from each machine
adata_list = []
for machine_name in cfg.machines:
    stage5_path = cfg.stage_dir(5, machine_name) / "adata_pooled.h5ad"
    validate_stage_inputs([stage5_path], f"Stage 6 ({machine_name})")
    
    adata_m = load_adata(stage5_path)
    adata_m.obs["machine"] = machine_name
    adata_list.append(adata_m)
    print(f"  {machine_name}: {adata_m.n_obs} cells, groups: {adata_m.obs['cluster_group'].unique().tolist()}")

# Merge all machines
adata = ad.concat(adata_list, join="outer")
adata.obs_names_make_unique()
print(f"\nMerged AnnData: {adata.n_obs} cells × {adata.n_vars} features")
print(f"Meta-cluster groups: {sorted(adata.obs['cluster_group'].unique().tolist())}")

stage6_dir = cfg.ensure_stage_dir(6)
candidates_dir = stage6_dir / "candidates"
candidates_dir.mkdir(parents=True, exist_ok=True)

[Stage 6 (machine_A)] All required inputs found ✓
  machine_A: 59102 cells, groups: ['Unspecific']

Merged AnnData: 59102 cells × 25 features
Meta-cluster groups: ['Unspecific']


## Run Subclustering at Multiple Resolutions

For each meta-cluster group, run Leiden at a range of resolutions and generate comparison plots.

In [ ]:
# Generate resolution range from config
sub_cfg = cfg.subclustering
resolutions = np.arange(
    sub_cfg.resolution_min,
    sub_cfg.resolution_max + sub_cfg.resolution_step / 2,
    sub_cfg.resolution_step,
).round(2).tolist()
print(f"Resolution range: {resolutions}")

meta_groups = sorted(adata.obs["cluster_group"].unique().tolist())
selected_markers = cfg.selected_markers

# Store all results: {group_name: {resolution: adata_subset}}
all_results = {}
# Summary for widget: {group_name: {resolution: n_subclusters}}
group_summary = {}

for group_name in meta_groups:
    print(f"\n{'='*50}")
    print(f"Group: {group_name}")
    print(f"{'='*50}")

    results = subcluster_at_resolutions(
        adata, group_name, resolutions,
        group_col="cluster_group",
        n_neighbors=cfg.clustering.n_neighbors,
        n_pcs=cfg.clustering.n_pcs,
    )
    all_results[group_name] = results

    # Generate candidate plots
    group_dir = candidates_dir / group_name.replace(" ", "_")
    group_dir.mkdir(parents=True, exist_ok=True)

    res_summary = {}
    for res, adata_sub in results.items():
        n_sub = adata_sub.obs["sub_cluster"].nunique()
        res_summary[res] = n_sub

        # Compute means for this subclustering
        sub_means = pd.DataFrame(
            adata_sub.X if not hasattr(adata_sub.X, "toarray") else adata_sub.X.toarray(),
            columns=selected_markers,
        )
        sub_means["sub_cluster"] = adata_sub.obs["sub_cluster"].values
        sub_means = sub_means.groupby("sub_cluster").mean()

        x_labels = sub_means.index.tolist()
        y_labels = selected_markers

        plot_dotplot(
            sub_means, x_labels, y_labels,
            save_path=group_dir / f"res_{res:.2f}_dotplot.svg",
            title=f"{group_name} – res={res:.2f} ({n_sub} subclusters)",
            show=False,
        )
        plot_heatmap(
            sub_means, x_labels, y_labels,
            save_path=group_dir / f"res_{res:.2f}_heatmap.svg",
            title=f"{group_name} – res={res:.2f} ({n_sub} subclusters)",
            show=False,
        )

    group_summary[group_name] = res_summary

print("\n✓ All candidate subclusters generated.")

Resolution range: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]

Group: Unspecific
  Subclustering 'Unspecific' at resolution=0.10


2026-02-17 01:35:45.284948: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-02-17 01:35:45.285054: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
/home/hp/projects/multiplex_imaging_analysis/pipeline/clustering.py:260: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(subset, resolution=resolution, key_added="sub_cluster")


    → 5 sub-clusters, 59102 cells
  Subclustering 'Unspecific' at resolution=0.20
    → 7 sub-clusters, 59102 cells
  Subclustering 'Unspecific' at resolution=0.30
    → 13 sub-clusters, 59102 cells
  Subclustering 'Unspecific' at resolution=0.40
    → 17 sub-clusters, 59102 cells
  Subclustering 'Unspecific' at resolution=0.50
    → 17 sub-clusters, 59102 cells
  Subclustering 'Unspecific' at resolution=0.60
    → 20 sub-clusters, 59102 cells
  Subclustering 'Unspecific' at resolution=0.70
    → 23 sub-clusters, 59102 cells
  Subclustering 'Unspecific' at resolution=0.80
    → 26 sub-clusters, 59102 cells
  Subclustering 'Unspecific' at resolution=0.90
    → 27 sub-clusters, 59102 cells
  Subclustering 'Unspecific' at resolution=1.00


## Visual Comparison

Review the candidate plots saved in `stage6_subclustering/candidates/<group>/` to compare resolutions. The dot plots and heatmaps above are saved as SVGs for detailed inspection.

In [ ]:
# Summary table of subclusters per resolution
print("Subclusters per resolution:\n")
summary_rows = []
for group_name, res_map in sorted(group_summary.items()):
    for res, n_sub in sorted(res_map.items()):
        summary_rows.append({"Group": group_name, "Resolution": res, "Subclusters": n_sub})
display(pd.DataFrame(summary_rows))

## Select Best Resolution per Group

Use the widget below to select the optimal resolution for each meta-cluster group.

In [ ]:
# Store selected resolutions
selected_resolutions = {}

def on_save_selections(selections):
    global selected_resolutions
    selected_resolutions = selections

picker = create_batch_resolution_picker(group_summary, on_save=on_save_selections)
display(picker)

## Apply Selected Resolutions & Build Final Output

Run this after selecting resolutions above and clicking "Save All Selections".

In [ ]:
if not selected_resolutions:
    raise ValueError("No resolutions selected! Use the widget above first.")

print("Applying selected resolutions:")
for g, r in selected_resolutions.items():
    print(f"  {g}: resolution={r:.2f}")

# Build final sub_cluster column on the merged adata
adata.obs["sub_cluster"] = ""
adata.obs["final_label"] = ""

for group_name, res in selected_resolutions.items():
    # Get the pre-computed subclustered AnnData for this resolution
    adata_sub = all_results[group_name][res]

    # Map sub_cluster back to main adata
    mask = adata.obs["cluster_group"] == group_name

    # The sub adata indices correspond to the masked adata
    sub_indices = adata.obs.index[mask]
    adata.obs.loc[sub_indices, "sub_cluster"] = adata_sub.obs["sub_cluster"].values

    # Final label = group:subcluster
    adata.obs.loc[sub_indices, "final_label"] = (
        group_name + ":" + adata_sub.obs["sub_cluster"].astype(str).values
    )

# For cells not in any group (shouldn't happen, but safety)
unlabelled = adata.obs["final_label"] == ""
if unlabelled.any():
    adata.obs.loc[unlabelled, "final_label"] = "Unassigned"
    adata.obs.loc[unlabelled, "sub_cluster"] = "0"

print(f"\nFinal label distribution:")
display(adata.obs["final_label"].value_counts())

## Final Heatmap & Save

In [ ]:
# Final heatmap across all subclusters
final_means = pd.DataFrame(
    adata.X if not hasattr(adata.X, "toarray") else adata.X.toarray(),
    columns=selected_markers,
)
final_means["final_label"] = adata.obs["final_label"].values
final_means = final_means.groupby("final_label").mean()

plot_heatmap(
    final_means,
    x_labels=final_means.index.tolist(),
    y_labels=selected_markers,
    save_path=stage6_dir / "final_heatmap.svg",
    title="Final Sub-cluster Heatmap (All Machines)",
)

plot_dotplot(
    final_means,
    x_labels=final_means.index.tolist(),
    y_labels=selected_markers,
    save_path=stage6_dir / "final_dotplot.svg",
    title="Final Sub-cluster Dot Plot (All Machines)",
)

In [ ]:
# Save final AnnData
save_adata(adata, stage6_dir / "final_adata.h5ad")

# Save per-ROI CSVs with full lineage columns
# Columns: Cell Id, Nuc X, Nuc Y Inv, region, machine,
#           initial_cluster (leiden), pooled_group (cluster_group),
#           sub_cluster, final_label
adata.obs.rename(columns={
    "leiden": "initial_cluster",
    "cluster_group": "pooled_group",
}, inplace=True)

roi_dir = stage6_dir / "roi_assignments"
save_roi_assignments(adata, roi_dir, region_col="unique_region")

print(f"\n✓ Stage 6 complete!")
print(f"  Outputs: {stage6_dir}")
print(f"  Final lineage columns: initial_cluster, pooled_group, sub_cluster, final_label")

## Optional: Spatial Visualisation per ROI

Generate spatial scatter plots for individual ROIs, colored by final sub-cluster label.

In [ ]:
spatial_dir = stage6_dir / "spatial_plots"
spatial_dir.mkdir(parents=True, exist_ok=True)

for region in sorted(adata.obs["unique_region"].unique()):
    region_mask = adata.obs["unique_region"] == region
    adata_region = adata[region_mask]
    safe_name = str(region).replace("/", "_").replace("\\", "_")
    plot_spatial(
        adata_region,
        save_path=spatial_dir / safe_name,
        cluster_col="final_label",
        title=f"Spatial: {region}",
        show=False,
    )

print(f"✓ Spatial plots saved to {spatial_dir}")